# 2️⃣ Flood Detection — Prithvi-Tiny + Logical Ensemble

**What this does:**
Meets official requirements by training IBM Prithvi (Tiny) natively at 512x512 with TTA enabled.
At the end, it merges its predictions with Notebook 1 (SMP UNet++) using Logical OR for extreme Flood Recall.




## Cell 1 — Install & Imports




In [ ]:
!pip install -q "numpy>=2.2,<2.3" "scipy>=1.14" terratorch rasterio albumentations

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
import rasterio

import terratorch
from terratorch.tasks import SemanticSegmentationTask
import albumentations
from albumentations.pytorch import ToTensorV2
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()



## Cell 2 — Config & Model




In [ ]:
DATA_ROOT = '/kaggle/input/anrfaisehack-theme-1-phase2/data/'
OUT_DIR = '/kaggle/working/'
PREDICT_PRITHVI_DIR = os.path.join(OUT_DIR, "prithvi_predictions")
PREDICT_SMP_DIR = os.path.join(OUT_DIR, "smp_predictions")
os.makedirs(PREDICT_PRITHVI_DIR, exist_ok=True)

kwargs=dict(
    backbone_pretrained=True, backbone='prithvi_eo_v2_tiny_tl',
    backbone_bands=[1,2,3,4,5,6], backbone_num_frames=1,
    decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
    num_classes=3, rescale=True, necks=[
        dict(name="SelectIndices", indices=[2, 5, 8, 11]),
        dict(name="ReshapeTokensToImage", effective_time_dim=1)
    ]
)
model = SemanticSegmentationTask(
    model_args=kwargs, loss='dice', lr=6e-5, optimizer="AdamW", 
    class_weights=[0.05, 0.60, 0.35],
    model_factory="EncoderDecoderFactory"
)



## Cell 3 — Train Prithvi 




In [ ]:
dm = terratorch.datamodules.GenericNonGeoSegmentationDataModule(
    batch_size=4, num_workers=2, num_classes=3,
    train_data_root=DATA_ROOT+'image', train_label_data_root=DATA_ROOT+'label',
    val_data_root=DATA_ROOT+'image', val_label_data_root=DATA_ROOT+'label',
    train_split=DATA_ROOT+'split/train.txt', val_split=DATA_ROOT+'split/val.txt',
    img_grep='*image.tif', label_grep='*label.tif',
    train_transform=[albumentations.D4(), ToTensorV2()],
    val_transform=[ToTensorV2()],
    means=[841.1, 371.6, 1734.1, 1588.3, 1742.8, 1218.5],
    stds=[473.7, 170.3, 623.0, 612.8, 564.5, 528.0]
)

trainer = pl.Trainer(
    accelerator="cuda", devices=1, precision='bf16-mixed', max_epochs=40,
    callbacks=[ModelCheckpoint(dirpath=OUT_DIR, filename="prithvi", save_top_k=1, monitor="val/mIoU")]
)
trainer.fit(model, datamodule=dm)



## Cell 4 — Prithvi Predict with TTA




In [ ]:
model.eval()
model.cuda()
predict_files = sorted(Path(DATA_ROOT + 'prediction/image/').glob("*.tif"))

for p in predict_files:
    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        meta = src.meta.copy()
        
    for b in range(6): raw[b] = (raw[b] - dm.means[b]) / dm.stds[b]
    raw = np.nan_to_num(raw, nan=0.0)
    
    x = torch.from_numpy(raw).unsqueeze(0).float().cuda()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        # Primitive TTA
        p1 = F.softmax(model(x)['output'], dim=1)
        p2 = torch.flip(F.softmax(model(torch.flip(x, dims=[3]))['output'], dim=1), dims=[3])
        probs = ((p1 + p2) / 2.0).squeeze(0).cpu().numpy()
        
    pred = np.argmax(probs, axis=0).astype("int16")
    meta.update({"count": 1, "dtype": "int16", "nodata": -1})
    with rasterio.open(os.path.join(PREDICT_PRITHVI_DIR, p.name), "w", **meta) as dst:
        dst.write(pred, 1)

print("✅ Saved Prithvi Predictions!")



## Cell 5 — 🌟 THE ULTIMATE ENSEMBLE




In [ ]:
def mask_to_rle(mask):
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

rows = []
for p_path in sorted(Path(PREDICT_PRITHVI_DIR).glob("*.tif")):
    s_path = Path(PREDICT_SMP_DIR) / p_path.name
    
    with rasterio.open(p_path) as s1: mask_p = s1.read(1)
        
    # Check if SMP prediction exists (if you ran Notebook 1)
    if s_path.exists():
        with rasterio.open(s_path) as s2: mask_s = s2.read(1)
        ensemble = ((mask_p == 1) | (mask_s == 1)).astype(np.uint8)
    else:
        print(f"Warning: SMP prediction {s_path.name} not found! Using Prithvi only.")
        ensemble = (mask_p == 1).astype(np.uint8)
        
    rle = mask_to_rle(ensemble)
    rows.append({"id": p_path.name.replace("_image.tif", ""), "rle_mask": rle if rle.strip() else "0 0"})

df = pd.DataFrame(rows)
df.to_csv("submission_ensemble.csv", index=False)
print(f"🎉 ENSEMBLE COMPLETE. Saved {len(df)} images to submission_ensemble.csv")
